[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/relativity/thomas_precession/thomas_precession.ipynb)

# Thomas Precession and the Muon's Anomalous Magnetic Moment

This example computes how a spinning particle's axis turns when its velocity changes direction without any external torque, and extends that kinematics to spin precession in a relativistic storage ring. Composing two Lorentz boosts in different directions leaves a residual spatial rotation behind, so carrying a particle's frame around any closed loop in hyperbolic velocity space turns its spatial axes by the hyperbolic area enclosed by the loop, an effect called Thomas precession. When the particle also carries a magnetic moment in an electromagnetic field, the muon anomalous-moment storage-ring experiments measure the rate at which the spin runs ahead of the orbital momentum, cancelling the contribution of focusing electric fields at a single magic momentum.

In spacetime geometric algebra, a four-velocity is modelled as a unit timelike antivector obtained by boosting the laboratory time direction `boost >> time`, and a particle's spin is modelled as an intrinsic antibivector patch of its spatial turning plane. Taking the square root `(head * tail.inverse()).square_root()` gives the direct boost from velocity `tail` to velocity `head`, so multiplying those step boosts transports a spin without torque around any velocity loop. In an electromagnetic bivector `field`, pairing `field & velocity` extracts the Lorentz four-force and wedging with `rest_slice = -velocity.dual()` builds a single frame-rate bivector `(1 + anomaly) * field - anomaly * (rest_slice ^ force)` that drives both the velocity and the spin together.

In relativistic three-vector and four-tensor index notation, torque-free velocity changes induce the Thomas precession rate $\boldsymbol{\omega}_T = -\frac{\gamma^2}{\gamma + 1}\frac{\mathbf{v} \times \mathbf{a}}{c^2}$ on a rest-frame spin vector, while electromagnetic evolution splits into separate worldline and spin equations, $\frac{du^\mu}{d\tau} = \frac{q}{m}F^\mu{}_\nu u^\nu$ and the Bargmann–Michel–Telegdi equation $\frac{ds^\mu}{d\tau} = \frac{q}{m}\left(\frac{g}{2}F^\mu{}_\nu s^\nu + \left(\frac{g}{2} - 1\right)u^\mu F_{\nu\lambda}u^\nu s^\lambda\right)$ subject to $u_\mu s^\mu = 0$.

In [ ]:
# Install numga and its examples when running in Colab.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from itertools import islice

import numpy as np

from numga import NumpyContext, concatenate, stack
from numga.algebras import STA
from examples.relativity.thomas_precession import render
from examples.util.animation import save_figure, show_states

ga = STA
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                         # Scalar+
Bivector = ga.gatype.bivector()                     # Bivector+
Spin = ga.gatype.antibivector()                     # Antibivector-
Antivector = ga.gatype.antivector()                 # Antivector-
Rotor = ga.gatype.rotor()                           # Rotor+

# Unit future-directed four-velocity of the laboratory observer.
time = -mv.t.dual()                                 # [] Antivector-
frame_duration_ms = 40

## Wigner rotation from two boosts

A rocket starting from rest boosts along its own horizontal axis by a chosen rapidity and then boosts along its own transverse axis by the same rapidity. Undoing the direct boost from rest to the resulting four-velocity via `step_boost(final_velocity, time) * rocket_boost` fixes the laboratory time direction and isolates a pure spatial rotor called the Wigner rotation.

In matrix Lorentz notation, the product of two non-collinear symmetric boost matrices factors uniquely by polar decomposition into a symmetric boost matrix and an orthogonal spatial rotation, $\Lambda_y(\eta)\Lambda_x(\eta) = B(\mathbf{v})\,R(\theta)$ with $\tan(\theta / 2) = -\tanh^2(\eta / 2)$.

In [ ]:
n_boost_frames = 120
min_rapidity = 0.8
max_rapidity = 2.0
n_edge_samples = 31

In [ ]:
def pure_boost(direction: Antivector, rapidity: np.ndarray) -> Rotor:
    """The spacetime plane spanned by `direction` (`[...] Antivector-`) and `time` exponentiates into a pure boost rotor (`[...] Rotor+`) of `rapidity`."""
    plane = (direction & time).dual()                                           # [...] Bivector+
    return (plane * (rapidity / 2)).exp()                                       # [...] Rotor+


def step_boost(tail: Antivector, head: Antivector) -> Rotor:
    """The geometric ratio `head * tail.inverse()` has a square-root rotor (`[...] Rotor+`) that boosts four-velocity `tail` directly to `head`."""
    return (head * tail.inverse()).square_root()                                # [...] Rotor+

In [ ]:
def geodesic_velocities(tail: Antivector, head: Antivector, n_samples: int) -> Antivector:
    """Fractional powers of the pure boost from `tail` to `head` sample `n_samples` four-velocities (`[n_samples] Antivector-`) along their hyperbolic geodesic."""
    fractions = np.linspace(0, 1, n_samples, endpoint=False)                    # [n_samples]
    steps = (step_boost(tail, head).log() * fractions).exp()                    # [n_samples] Rotor+
    return steps >> tail                                                        # [n_samples] Antivector-


def wigner_rotor(rapidity: np.ndarray) -> Rotor:
    """Two perpendicular boosts of equal `rapidity` followed by the direct boost back to rest leave a pure spatial Wigner rotor (`[...] Rotor+`)."""
    rocket_boost = pure_boost(mv.x.dual(), rapidity) * pure_boost(mv.y.dual(), rapidity)   # [...] Rotor+
    final_velocity = rocket_boost >> time                                       # [...] Antivector-
    return step_boost(final_velocity, time) * rocket_boost                      # [...] Rotor+

In [ ]:
def two_boost_states(rapidities: np.ndarray, n_samples: int) -> Iterator[tuple[Antivector, Antivector, Antivector]]:
    """Yields the geodesic velocity triangle (`[3 * n_samples] Antivector-`), its three corner velocities (`[3] Antivector-`), and the rest-frame spatial axis (`[3] Antivector-`) at each rapidity."""
    x_axis = mv.x.dual()                                                        # [] Antivector-
    y_axis = mv.y.dual()                                                        # [] Antivector-
    for rapidity in rapidities:
        first_boost = pure_boost(x_axis, rapidity)                              # [] Rotor+
        rocket_boost = first_boost * pure_boost(y_axis, rapidity)               # [] Rotor+
        corner_velocity = first_boost >> time                                   # [] Antivector-
        final_velocity = rocket_boost >> time                                   # [] Antivector-
        loop = concatenate([
            geodesic_velocities(time, corner_velocity, n_samples),
            geodesic_velocities(corner_velocity, final_velocity, n_samples),
            geodesic_velocities(final_velocity, time, n_samples),
        ])                                                                      # [3 * n_samples] Antivector-
        corner_unboost = step_boost(corner_velocity, time)                      # [] Rotor+
        final_unboost = step_boost(final_velocity, time)                        # [] Rotor+
        axes = stack([
            x_axis,
            corner_unboost >> (first_boost >> x_axis),
            final_unboost >> (rocket_boost >> x_axis),
        ])                                                                      # [3] Antivector-
        yield loop, stack([time, corner_velocity, final_velocity]), axes


swell = (1 - np.cos(np.linspace(0, 2 * np.pi, n_boost_frames, endpoint=False))) / 2
rapidities = min_rapidity + (max_rapidity - min_rapidity) * swell               # [n_boost_frames]
wigner_rotors = wigner_rotor(rapidities)                                        # [n_boost_frames] Rotor+
boost_history = show_states(
    two_boost_states(rapidities, n_edge_samples),
    render.two_boosts_animation,
    name="thomas_two_boosts",
    duration_ms=frame_duration_ms,
);

Stereographic projection onto the Poincaré disc places rest at the centre, light speed on the grey outer circle, and pure boosts along circular arcs that meet the boundary at right angles. As the rapidity swells and shrinks, the blue hyperbolic triangle grows and contracts while the orange arrows track the rocket's horizontal axis at each corner as read in the laboratory rest slice; at the final corner, the angular gap between the orange arrow and the faint grey reference arrow equals the hyperbolic area of the shaded triangle.

## Thomas precession on circular orbits

A particle moving at constant speed around a circle traces a closed circle of constant rapidity on the velocity hyperboloid. Successive step boosts `step_boost(tail, head)` transport the gyroscope's intrinsic spin patch from each four-velocity to the next without torque, accumulating tiny Wigner rotations that turn the spin opposite to the orbital motion.

In three-vector special-relativity notation, integrating $\boldsymbol{\omega}_T = -\frac{\gamma^2}{\gamma + 1}\frac{\mathbf{v} \times \mathbf{a}}{c^2}$ around one revolution at Lorentz factor $\gamma$ gives a net retrograde spin angle of $-2\pi(\gamma - 1)$, matching the hyperbolic area of the velocity cap bounded by the orbit.

In [ ]:
orbit_gammas = np.array([1.25, 1.5, 1.75])
n_orbit_steps = 448
n_orbit_stride = 4

In [ ]:
def lab_spin(velocity: Antivector, spin: Spin) -> Spin:
    """The direct boost from `velocity` (`[...] Antivector-`) back to `time` carries `spin` into the laboratory rest slice (`[...] Spin-`)."""
    boost_to_rest = step_boost(velocity, time)                                  # [...] Rotor+
    return boost_to_rest >> spin                                                # [...] Spin-


def carried_spins(velocities: Antivector, spin: Spin) -> Iterator[Spin]:
    """Yields `spin` (`[...] Spin-`) transported without torque along `velocities` (`[n_steps + 1, ...] Antivector-`) by successive pure boosts."""
    yield spin
    for tail, head in zip(velocities[:-1], velocities[1:]):
        step = step_boost(tail, head)                                           # [...] Rotor+
        spin = step >> spin                                                     # [...] Spin-
        yield spin

In [ ]:
def orbit_velocities(gammas: np.ndarray, n_steps: int) -> Antivector:
    """Counterclockwise spatial directions boosted to Lorentz factors `gammas` trace circular velocity loops (`[n_steps + 1, n_orbits] Antivector-`)."""
    n_orbits = len(gammas)
    phases = np.linspace(0, 2 * np.pi, n_steps + 1)                             # [n_steps + 1]
    turns = (mv.xy * (-phases[:, None] / 2)).exp()                              # [n_steps + 1, 1] Rotor+
    directions = turns >> mv.y.dual()                                           # [n_steps + 1, 1] Antivector-
    boosts = pure_boost(directions, np.arccosh(gammas))                         # [n_steps + 1, n_orbits] Rotor+
    return boosts >> time                                                       # [n_steps + 1, n_orbits] Antivector-


def orbit_states(velocities: Antivector, spin: Spin, n_stride: int) -> Iterator[tuple[Antivector, Spin]]:
    """Yields every `n_stride`-th orbital four-velocity and laboratory-read spin (`[n_orbits] Antivector-`, `[n_orbits] Spin-`)."""
    for velocity, carried_spin in islice(zip(velocities, carried_spins(velocities, spin)), None, None, n_stride):
        yield velocity, lab_spin(velocity, carried_spin)


velocities = orbit_velocities(orbit_gammas, n_orbit_steps)                      # [n_orbit_steps + 1, 3] Antivector-
gyroscope_spin = mv.yz.dual()                                                   # [] Spin-
initial_orbit_spins = step_boost(time, velocities[0]) >> gyroscope_spin         # [3] Spin-
orbit_history = show_states(
    orbit_states(velocities, initial_orbit_spins, n_orbit_stride),
    render.orbits_animation,
    velocities,
    name="thomas_orbits",
    duration_ms=frame_duration_ms,
);

Concentric green, blue, and purple circles on the Poincaré disc trace circular orbits at Lorentz factors 1.25, 1.5, and 1.75, with faint grey arrows marking the initial laboratory spin direction on each ring. Over one counterclockwise revolution, the three carried spin arrows lag clockwise by one quarter, one half, and three quarters of a turn.

## Gyroscope bouncing in a square box

Velocity-space holonomy depends only on the closed path in velocity space rather than smooth circular acceleration. Inside a square box, a gyroscope flies at constant velocity between adjacent wall midpoints so nothing turns during free flight, while each elastic wall bounce reverses the normal spatial momentum and lifts the interpolated momentum onto the velocity hyperboloid via `hyperboloid_velocity`.

In [ ]:
box_half_width = 1.0
box_momentum = 0.8
n_flight_frames = 13
n_bounce_steps = 12
n_loops = 3

In [ ]:
def hyperboloid_velocity(spatial: Antivector) -> Antivector:
    """Adding the unit-mass energy component along `time` lifts spatial four-momentum `spatial` (`[...] Antivector-`) onto the unit velocity hyperboloid."""
    return spatial + time * (1 + spatial.squared()).square_root()               # [...] Antivector-


def box_trajectory(half_width: float, momentum: float, n_flight: int, n_bounce: int) -> tuple[Antivector, Antivector]:
    """Straight wall-to-wall flights and wall bounces build one diamond round trip of positions and four-velocities (`[4 * (n_flight + n_bounce)] Antivector-`)."""
    quarter_turns = (mv.xy * (-np.arange(-1, 5) * np.pi / 4)).exp()             # [6] Rotor+
    walls = quarter_turns >> (half_width * mv.x.dual())                         # [6] Antivector-
    corner_momenta = quarter_turns >> (momentum * (mv.x.dual() + mv.y.dual()))  # [6] Antivector-
    wall_starts, wall_ends = walls[:-2], walls[1:-1]                            # [4] Antivector-, Antivector-
    flights = wall_starts + (wall_ends - wall_starts) * np.linspace(0, 1, n_flight)[:, None]   # [n_flight, 4] Antivector-
    incoming_momenta, outgoing_momenta = corner_momenta[1:-1], corner_momenta[2:]              # [4] Antivector-, Antivector-
    bounces = hyperboloid_velocity(
        incoming_momenta + (outgoing_momenta - incoming_momenta) * np.linspace(0, 1, n_bounce)[:, None]
    )                                                                           # [n_bounce, 4] Antivector-
    flight_velocities = hyperboloid_velocity(incoming_momenta).broadcast_to((n_flight, 4))     # [n_flight, 4] Antivector-
    positions = concatenate([
        segment[:, wall] for wall in range(4) for segment in (flights, wall_ends.broadcast_to((n_bounce, 4)))
    ])                                                                          # [4 * (n_flight + n_bounce)] Antivector-
    velocities = concatenate([
        segment[:, wall] for wall in range(4) for segment in (flight_velocities, bounces)
    ])                                                                          # [4 * (n_flight + n_bounce)] Antivector-
    return positions, velocities

In [ ]:
def box_states(positions: Antivector, velocities: Antivector, spin: Spin) -> Iterator[tuple[Antivector, Spin]]:
    """Yields each particle position and laboratory-read spin (`[] Antivector-`, `[] Spin-`) along the box trajectory."""
    for position, velocity, carried_spin in zip(positions, velocities, carried_spins(velocities, spin)):
        yield position, lab_spin(velocity, carried_spin)


single_positions, single_velocities = box_trajectory(
    box_half_width, box_momentum, n_flight_frames, n_bounce_steps
)
positions = concatenate([single_positions] * n_loops)
box_velocities = concatenate([single_velocities] * n_loops)
initial_box_spin = step_boost(time, box_velocities[0]) >> gyroscope_spin        # [] Spin-
wall_turns = (mv.xy * (-np.arange(4) * np.pi / 4)).exp()                        # [4] Rotor+
wall_midpoints = wall_turns >> (box_half_width * mv.x.dual())                   # [4] Antivector-
box_history = show_states(
    box_states(positions, box_velocities, initial_box_spin),
    render.box_animation,
    box_half_width,
    wall_midpoints,
    name="thomas_box",
    duration_ms=frame_duration_ms,
);

Along the faint grey diamond connecting the midpoints of the dark square walls, the orange spin arrow stays fixed on every straight segment and steps clockwise at each wall reflection, accumulating one velocity-square holonomy angle per round trip.

## Spin precession in a muon storage ring

In a uniform magnetic field, a particle with gyromagnetic factor two turns its spin and its momentum at the same rate, whereas a nonzero magnetic anomaly makes the spin pull ahead of the momentum. Because `(rest_slice ^ force)` acts identically to `field` on `velocity` while isolating the velocity-change plane on orthogonal spatial axes, the single bivector `(1 + anomaly) * field - anomaly * (rest_slice ^ force)` integrates the Lorentz orbit and the anomalous spin precession simultaneously through one rotor frame `frame`.

In four-tensor index notation, relativistic spin-orbit evolution separates into the Lorentz force equation $\frac{du^\mu}{d\tau} = \frac{q}{m}F^\mu{}_\nu u^\nu$ and the Bargmann–Michel–Telegdi equation $\frac{ds^\mu}{d\tau} = \frac{q}{m}\left(\frac{g}{2}F^\mu{}_\nu s^\nu + \left(\frac{g}{2} - 1\right)u^\mu F_{\nu\lambda}u^\nu s^\lambda\right)$.

In [ ]:
muon_anomaly = 0.1
magic_gamma = float(np.sqrt(1 + 1 / muon_anomaly))
ring_magnetic = 1.0
ring_electric = 0.0
n_ring_turns = 3
n_ring_turn_steps = 300
n_ring_stride = 6
n_ring_samples = 127
muon_spin = mv.zx.dual()                                                        # [] Spin-

In [ ]:
def bmt_generator(field: Bivector, velocity: Antivector, anomaly: float) -> Bivector:
    """Electromagnetic `field` (`[...] Bivector+`) and magnetic `anomaly` combine at `velocity` into the Bargmann–Michel–Telegdi frame rate (`[...] Bivector+`)."""
    force = field & velocity                                                    # [...] Vector+
    # The particle's instantaneous rest slice pairs to unity with its four-velocity.
    rest_slice = -velocity.dual()                                               # [...] Vector+
    # On `velocity` itself `(rest_slice ^ force)` matches `field`, cancelling `anomaly` from the worldline acceleration.
    return (1 + anomaly) * field - anomaly * (rest_slice ^ force)               # [...] Bivector+


def ring_field(event: Antivector, magnetic: float, electric: np.ndarray) -> Bivector:
    """Vertical `magnetic` bending and radial `electric` focusing form the storage-ring field bivector (`[...] Bivector+`) at `event`."""
    position = event - (mv.t & event) * time                                    # [...] Antivector-
    outward = position / position.antinorm()                                    # [...] Antivector-
    return -magnetic * mv.xy + electric * (outward & time).dual()               # [...] Bivector+

In [ ]:
def bmt_step(
    event: Antivector,
    frame: Rotor,
    magnetic: float,
    electric: np.ndarray,
    anomaly: float,
    proper_step: np.ndarray,
) -> tuple[Antivector, Rotor]:
    """Midpoint proper-time integration advances `event` (`[...] Antivector-`) and `frame` (`[...] Rotor+`) by `proper_step` in the storage-ring field."""
    velocity = frame >> time                                                    # [...] Antivector-
    rate = bmt_generator(ring_field(event, magnetic, electric), velocity, anomaly)   # [...] Bivector+
    halfway_frame = (rate * (proper_step / 4)).exp() * frame                    # [...] Rotor+
    halfway_event = event + velocity * (proper_step / 2)                        # [...] Antivector-
    halfway_velocity = halfway_frame >> time                                    # [...] Antivector-
    halfway_rate = bmt_generator(ring_field(halfway_event, magnetic, electric), halfway_velocity, anomaly)   # [...] Bivector+
    next_event = event + halfway_velocity * proper_step                         # [...] Antivector-
    next_frame = (halfway_rate * (proper_step / 2)).exp() * frame               # [...] Rotor+
    return next_event, next_frame


def bmt_trajectory(
    event: Antivector,
    frame: Rotor,
    magnetic: float,
    electric: np.ndarray,
    anomaly: float,
    proper_step: np.ndarray,
) -> Iterator[tuple[Antivector, Rotor]]:
    """Yields successive particle events and Lorentz frames (`[...] Antivector-`, `[...] Rotor+`) integrated by `proper_step`."""
    while True:
        yield event, frame
        event, frame = bmt_step(event, frame, magnetic, electric, anomaly, proper_step)

In [ ]:
def ring_radius(gamma: np.ndarray, magnetic: float, electric: np.ndarray) -> np.ndarray:
    """Radial force balance between `magnetic` bending and `electric` focusing sets the circular orbit radius at Lorentz factor `gamma`."""
    speed = np.sqrt(1 - 1 / gamma**2)
    return gamma * speed**2 / (speed * magnetic - electric)


def ring_start(gamma: np.ndarray, magnetic: float, electric: np.ndarray) -> tuple[Antivector, Rotor]:
    """Initial position on the horizontal axis and pure boost along the tangent initialize a circular storage-ring orbit (`[...] Antivector-`, `[...] Rotor+`)."""
    event = mv.x.dual() * ring_radius(gamma, magnetic, electric)                # [...] Antivector-
    frame = pure_boost(mv.y.dual(), np.arccosh(gamma))                          # [...] Rotor+
    return event, frame

In [ ]:
def ring_states(
    gamma: float,
    magnetic: float,
    electric: float,
    anomaly: float,
    muon_spin: Spin,
    n_turns: int,
    n_turn_steps: int,
    n_stride: int,
) -> Iterator[tuple[Antivector, Antivector, Spin]]:
    """Yields every `n_stride`-th orbital event, four-velocity, and laboratory-read spin (`[] Antivector-`, `[] Antivector-`, `[] Spin-`) across `n_turns` revolutions."""
    radius = ring_radius(gamma, magnetic, electric)
    proper_period = 2 * np.pi * radius / np.sqrt(gamma**2 - 1)
    event, frame = ring_start(gamma, magnetic, electric)
    states = bmt_trajectory(event, frame, magnetic, electric, anomaly, proper_period / n_turn_steps)
    for event, frame in islice(states, 0, n_turns * n_turn_steps + 1, n_stride):
        velocity = frame >> time                                                # [] Antivector-
        yield event, velocity, lab_spin(velocity, frame >> muon_spin)


radius = ring_radius(magic_gamma, ring_magnetic, ring_electric)
ring_turns = (mv.xy * (-np.linspace(0, 2 * np.pi, n_ring_samples, endpoint=False) / 2)).exp()   # [n_ring_samples] Rotor+
ring_points = ring_turns >> (radius * mv.x.dual())                              # [n_ring_samples] Antivector-
ring_history = show_states(
    ring_states(
        magic_gamma,
        ring_magnetic,
        ring_electric,
        muon_anomaly,
        muon_spin,
        n_ring_turns,
        n_ring_turn_steps,
        n_ring_stride,
    ),
    render.ring_animation,
    ring_points,
    name="thomas_storage_ring",
    duration_ms=frame_duration_ms,
);

With the magnetic anomaly exaggerated to 0.1 at the corresponding magic Lorentz factor, the orange spin arrow rotates counterclockwise faster than the grey tangent momentum arrow and completes almost one full extra revolution over three turns around the storage ring.

## Focusing electric fields and the magic momentum

Electrostatic quadrupoles that focus a storage-ring beam vertically introduce radial electric fields, which appear partly magnetic in the moving muon's rest frame and shift the spin precession rate. At one critical Lorentz factor where the square of the relativistic momentum equals the reciprocal of the magnetic anomaly, the boosted electric torque cancels the electric Thomas-precession contribution identically, leaving the anomalous precession rate independent of the focusing electric field.

In the three-vector notation of muon storage-ring experiments, the spin frequency relative to momentum reads as $\boldsymbol{\omega}_a = -\frac{q}{m}\left[a\,\mathbf{B} - \left(a - \frac{1}{\gamma^2 - 1}\right)\frac{\boldsymbol{\beta} \times \mathbf{E}}{c}\right]$, whose electric term vanishes at the magic Lorentz factor $\gamma_{\mathrm{magic}} = \sqrt{1 + 1/a}$.

In [ ]:
electric_fields = np.linspace(-0.3, 0.3, 7)
sweep_gammas = np.array([2.0, np.sqrt(1 + 1 / muon_anomaly)])
sweep_lab_duration = 10.0
n_sweep_steps = 200

In [ ]:
def anomalous_angle(velocity: Antivector, spin: Spin) -> Scalar:
    """Logarithmic comparison in the orbital plane measures the laboratory angle (`[...] Scalar+`) by which `spin` leads the forward momentum direction."""
    motion = velocity - (mv.t & velocity) * time                                # [...] Antivector-
    direction = motion / motion.antinorm()                                      # [...] Antivector-
    # Intrinsic spin patch aligned with the orbital tangent in the horizontal plane.
    forward_spin = direction.commutator(mv.xy).regressive(mv.z.dual())          # [...] Spin-
    return (forward_spin.inverse() * lab_spin(velocity, spin)).log() & mv.xy.dual()   # [...] Scalar+


def anomalous_rates(
    gammas: np.ndarray,
    magnetic: float,
    electric: np.ndarray,
    anomaly: float,
    muon_spin: Spin,
    lab_duration: float,
    n_steps: int,
) -> Scalar:
    """Proper-time integration over `lab_duration` returns the normalized anomalous precession rate (`[n_gammas, n_fields] Scalar+`) across Lorentz factors and electric fields."""
    n_gammas = len(gammas)
    n_fields = len(electric)
    event, frame = ring_start(gammas[:, None], magnetic, electric)              # [n_gammas, n_fields] Antivector-, Rotor+
    proper_step = lab_duration / (gammas[:, None] * n_steps)                    # [n_gammas, 1]
    *_, (event, frame) = islice(
        bmt_trajectory(event, frame, magnetic, electric, anomaly, proper_step),
        n_steps + 1,
    )
    angles = anomalous_angle(frame >> time, frame >> muon_spin)                 # [n_gammas, n_fields] Scalar+
    return angles / (anomaly * magnetic * lab_duration)                         # [n_gammas, n_fields] Scalar+


rate_ratios = anomalous_rates(
    sweep_gammas,
    ring_magnetic,
    electric_fields,
    muon_anomaly,
    muon_spin,
    sweep_lab_duration,
    n_sweep_steps,
)                                                                               # [2, 7] Scalar+
save_figure(render.magic_momentum_figure(electric_fields / ring_magnetic, rate_ratios), "thomas_magic");

Across radial electric fields from negative to positive, the grey curve at Lorentz factor 2 tilts strongly with electric field strength, whereas the orange curve at the magic Lorentz factor stays flat at unit normalized precession rate.

In [ ]:
# checks
# Two perpendicular boosts leave a pure spatial rotation matching the closed-form Wigner angle.
assert len(boost_history) == n_boost_frames
assert ((mv.t & ((wigner_rotors >> time) - time)).abs() < 1e-6).all()
wigner_angles = 2 * wigner_rotors.log().scalar_product(mv.xy)
assert ((wigner_angles + 2 * np.arctan(np.tanh(rapidities / 2) ** 2)).abs() < 1e-6).all()

# One circular velocity orbit turns each spin back by the enclosed hyperbolic cap area 2 * pi * (gamma - 1).
_, final_orbit_spins = orbit_history[-1]
orbit_turns = (mv.xy * (np.pi * (orbit_gammas - 1))).exp()
expected_orbit_spins = orbit_turns >> gyroscope_spin
assert ((final_orbit_spins - expected_orbit_spins).antinorm() < 5e-3).all()

# Three diamond round trips in the box turn the spin back by three velocity-square hyperbolic areas.
_, final_box_spin = box_history[-1]
square_area = 4 * (
    2 * box_momentum * np.arcsinh(box_momentum / np.sqrt(1 + box_momentum**2))
    - np.arctan(box_momentum**2 / np.sqrt(1 + 2 * box_momentum**2))
)
box_turn = (mv.xy * (n_loops * square_area / 2)).exp()
expected_box_spin = box_turn >> gyroscope_spin
assert ((final_box_spin - expected_box_spin).antinorm() < 5e-2).all()

# In the pure magnetic ring, three revolutions advance the laboratory spin by 6 * pi * anomaly * gamma.
_, _, initial_ring_spin = ring_history[0]
_, _, final_ring_spin = ring_history[-1]
ring_turn = (mv.xy * (-np.pi * n_ring_turns * muon_anomaly * magic_gamma)).exp()
expected_ring_spin = ring_turn >> initial_ring_spin
assert ((final_ring_spin - expected_ring_spin).antinorm() < 5e-3).all()

# Radial electric fields drop out at the magic Lorentz factor and follow the BMT slope off it.
low_speed = np.sqrt(1 - 1 / sweep_gammas[0] ** 2)
expected_low_rates = (
    1
    - (1 - 1 / (muon_anomaly * (sweep_gammas[0] ** 2 - 1)))
    * low_speed
    * electric_fields
    / ring_magnetic
)
assert ((rate_ratios[1] - 1).abs() < 1e-3).all()
assert ((rate_ratios[0] - expected_low_rates).abs() < 1e-3).all()